## Self notes — running log

A scratch notebook for personal observations: subtle Spark behaviours noticed while reading plans, debugging jobs, or skimming source. Each entry is a short note plus a tiny verifiable demo. Keep adding — newest at the bottom, or wherever fits.

> Conventions for new entries:
> - One markdown cell (the note) + one code cell (the proof) per item
> - Tiny demos only; no `data/` dependency unless unavoidable
> - Title the markdown with `### N — short topic`


## Setup


In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .appName("SelfNotes")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

SCRATCH = os.path.abspath("_scratch_11")
os.makedirs(SCRATCH, exist_ok=True)
print(f"Spark {spark.version} ready")


26/05/12 09:24:47 WARN Utils: Your hostname, maddipotis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.181 instead (on interface en0)
26/05/12 09:24:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/12 09:24:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 ready


### 1 — `SparkSession` has two flavours: classic and Connect

There are **two** `SparkSession` (and `Builder`) classes in PySpark, with identical-looking APIs:

| Module path | Mode | Talks to |
|---|---|---|
| `pyspark.sql.session.SparkSession` | **Classic** | A JVM started in-process (or via `spark-submit`) |
| `pyspark.sql.connect.session.SparkSession` | **Connect** | A remote Spark server via gRPC |

Why this matters in practice:

- Helpers that need to inject JARs into the JVM (e.g. `delta.configure_spark_with_delta_pip(builder)`) accept **only** the classic builder. Hand them a Connect builder and they throw with the type-mismatch message.
- `from pyspark.sql import SparkSession` in a fresh process returns classic — **unless** `SPARK_REMOTE` / `SPARK_CONNECT_MODE_ENABLED` is set, Databricks Connect is installed and active, or something in the environment reassigns `pyspark.sql` to `pyspark.sql.connect`.
- Once the session exists, `type(spark)` reveals which one you're holding. Builders show their mode in `type(...).__module__` (contains `connect` or not).

To force the classic path explicitly:

```python
from pyspark.sql.session import SparkSession   # not pyspark.sql.connect.session
os.environ.pop("SPARK_REMOTE", None)
os.environ.pop("SPARK_CONNECT_MODE_ENABLED", None)
```


In [2]:
# Show what we actually have
print(f"Session class : {type(spark).__module__}.{type(spark).__name__}")
print(f"Builder class : {type(SparkSession.builder).__module__}.{type(SparkSession.builder).__name__}")

is_connect = "connect" in type(SparkSession.builder).__module__
print(f"In Connect mode? {is_connect}")

# Environment hints that would flip you into Connect
for var in ("SPARK_REMOTE", "SPARK_CONNECT_MODE_ENABLED"):
    print(f"  {var} = {os.environ.get(var, '(not set)')}")


Session class : pyspark.sql.session.SparkSession
Builder class : pyspark.sql.session.Builder
In Connect mode? False
  SPARK_REMOTE = (not set)
  SPARK_CONNECT_MODE_ENABLED = (not set)


### 2 — Predicate pushdown: in-memory sources skip it, file sources still re-filter

Catalyst tries to push `WHERE` predicates down into the data source so fewer bytes ever leave disk. Two important nuances I keep forgetting:

**(a) In-memory sources can't push filters.** When the source is `ExistingRDD`, `LocalTableScan` (from `createDataFrame`), or `Range` (from `spark.range`), there is no scan layer to push into — the data is already in memory. The filter stays in the plan as a regular `Filter` node.

**(b) Even for file sources, the post-scan `Filter` does NOT disappear after pushdown.** The scan-level filter is **best effort**:
- Parquet pushdown uses footer min/max stats per row group. The scan can skip whole groups whose stats don't satisfy the predicate, but rows inside surviving groups still need to be re-checked. Catalyst keeps the `Filter` node above the scan to guarantee correctness.
- You'll see this in `df.explain()` as both `PushedFilters: [...]` on the `FileScan` AND a `Filter (...)` node above it. They're not redundant — they're complementary.

```
*(1) Project [customer_id#0, full_name#1, credit_score#3L]
  +- *(1) Filter (isnotnull(credit_score#3L) AND (credit_score#3L >= 700))   ← exact re-check
     +- *(1) ColumnarToRow                                                    ← vectorized → row format
        +- FileScan parquet [...]
             PushedFilters: [IsNotNull(credit_score), GreaterThanOrEqual(credit_score,700)]
             ReadSchema: struct<customer_id:string,full_name:string,credit_score:bigint>
```

What this means operationally:
- A `PushedFilters: []` line on a file scan is a red flag — your predicate isn't pushable (e.g. it uses a UDF, a non-deterministic expression, or a type the format can't compare).
- A `Filter` node sitting on top of a `LocalTableScan` / `Range` / `ExistingRDD` is normal — there's no pushdown opportunity there.


In [3]:
# (a) In-memory sources: filter has nowhere to push
print("─── Range source ───")
spark.range(1_000_000).filter("id > 500").explain(False)

print("\n─── LocalTableScan (from createDataFrame) ───")
spark.createDataFrame(
    [(i, f"name_{i}") for i in range(10)], "id INT, name STRING"
).filter("id > 5").explain(False)


─── Range source ───


== Physical Plan ==
*(1) Filter (id#0L > 500)
+- *(1) Range (0, 1000000, step=1, splits=12)



─── LocalTableScan (from createDataFrame) ───


== Physical Plan ==
*(1) Filter (isnotnull(id#4) AND (id#4 > 5))
+- *(1) Scan ExistingRDD[id#4,name#5]




In [4]:
# (b) Parquet source: filter pushes AND remains for correctness
PQ = os.path.join(SCRATCH, "customers.parquet")

(spark.createDataFrame(
    [(f"C{i:04d}", f"Name {i}", "NY" if i % 2 else "CA", 600 + (i % 250))
     for i in range(2000)],
    "customer_id STRING, full_name STRING, state STRING, credit_score INT")
   .write.mode("overwrite").parquet(PQ))

q = (spark.read.parquet(PQ)
     .filter(F.col("credit_score") >= 700)
     .select("customer_id", "full_name", "credit_score"))

print("Note both PushedFilters on the FileScan AND a Filter node above it:\n")
q.explain(False)


Note both PushedFilters on the FileScan AND a Filter node above it:

== Physical Plan ==
*(1) Filter (isnotnull(credit_score#23) AND (credit_score#23 >= 700))
+- *(1) ColumnarToRow
   +- FileScan parquet [customer_id#20,full_name#21,credit_score#23] Batched: true, DataFilters: [isnotnull(credit_score#23), (credit_score#23 >= 700)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/maddipotiganesh/Applications/apache-spark/_scratch_11/cust..., PartitionFilters: [], PushedFilters: [IsNotNull(credit_score), GreaterThanOrEqual(credit_score,700)], ReadSchema: struct<customer_id:string,full_name:string,credit_score:int>




In [5]:
# Predicate that CANNOT be pushed — a Python UDF — leaves PushedFilters empty
@F.udf("boolean")
def is_high(score):
    return score is not None and score >= 700

bad = (spark.read.parquet(PQ)
       .filter(is_high("credit_score"))
       .select("customer_id", "credit_score"))

print("PushedFilters is [] because the predicate is a Python UDF (opaque to Catalyst):\n")
bad.explain(False)


PushedFilters is [] because the predicate is a Python UDF (opaque to Catalyst):

== Physical Plan ==
*(2) Project [customer_id#32, credit_score#35]
+- *(2) Filter pythonUDF0#44: boolean
   +- BatchEvalPython [is_high(credit_score#35)#40], [pythonUDF0#44]
      +- *(1) ColumnarToRow
         +- FileScan parquet [customer_id#32,credit_score#35] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/maddipotiganesh/Applications/apache-spark/_scratch_11/cust..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:string,credit_score:int>




---

_Add new notes below. Suggested template:_

```
### N — title

Short note explaining the behaviour and why it matters.

Code cell: minimal demo that proves it.
```


In [6]:
# Future entries go here. Cleanup left out on purpose so the scratch dir
# survives between runs — delete _scratch_11 manually when needed.
# spark.stop()
